# NucleiSky2D: API Workflow Example

This notebook demonstrates how to align a cropped field-of-view (FOV) to a large reference image using the **NucleiSky2D** API. It follows the same steps as the core pipeline: load data, normalize pixel sizes for segmentation, segment nuclei, extract features, then run either the adaptive or manual matcher.

## Prerequisites
- `nucleisky2d` installed in your environment.
- A large reference image (e.g., stitched scan).
- A cropped image (e.g., FOV patch).
- Pixel size metadata (µm/px) for both images. You can pull this from TIFF metadata with `get_pixel_size_um_from_tiff`.

## Tips
- **Segmentation scale normalization** is important when the two images have different pixel sizes. The helper `scale_normalize_pair_for_segmentation` returns segmentation-scale images and effective pixel sizes.
- The **adaptive matcher** automatically chooses an order of matchers based on the number of nuclei; the **manual matcher** lets you override config (e.g., rotation range).


In [ ]:
# @title  Initialize Environment & Install Dependencies

import sys

# Metadata
current_version = "0.1.0"
notebook_name = "NucleiSky2D_API_Workflow_Example"

# ---------------------------------------------------------
# 1. Environment Detection & Installation (Colab Only)
# ---------------------------------------------------------

if 'google.colab' in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")

    # Incompatibilities due to the newest scikit-image==0.26.0 version
    !pip install -q cucim-cu12==26.4.0
    !pip install -q --upgrade nucleisky[all]

    from google.colab import drive
    drive.mount('/content/gdrive')

    print("✅ Colab setup done")

else:
    # Fallback for local environments
    print("✅ Local environment detected. Assuming dependencies are already installed.")


# ---------------------------------------------------------
# 2. GPU Check
# ---------------------------------------------------------
try:
    import torch
    if torch.cuda.is_available():
            print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠️  WARNING: No GPU detected. Segmentation with Cellpose or InstanSeg will be VERY SLOW on CPU.")
        if 'google.colab' in sys.modules:
            print("Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")
except ImportError:
    print("⚠️  WARNING: PyTorch not installed. Cellpose and InstanSeg segmentation will not work. If you want to use these features, please check the installation instructions.")

import numpy as np
import matplotlib.pyplot as plt
from tifffile import imread
from pathlib import Path
from skimage.segmentation import find_boundaries

# NucleiSky core imports
from nucleisky.nucleisky2d.pipeline import NucleiSky, run_adaptive_matching_and_export
from nucleisky.nucleisky2d.segmentation import segment_nuclei_dispatch
from nucleisky.nucleisky2d.features import (
    extract_nuclear_features,
    add_centroids_orig_px_columns,
    extract_centroids_um,
)
from nucleisky.nucleisky2d.preprocess import ij_percentile_normalize, scale_normalize_pair_for_segmentation
from nucleisky.nucleisky2d.config import DEFAULT_MATCHER_CONFIG
from nucleisky.nucleisky2d.visualization import show_alignment_original_and_rescaled
from nucleisky.nucleisky2d.io import get_pixel_size_um_from_tiff, save_nucleisky_transform
from nucleisky.nucleisky2d.export import export_aligned_imagej_stacks

print("✅ Imports successful.")


In [ ]:
# --- Helper utilities (optional) ---

def load_tiff(path: Path) -> np.ndarray:
    """Load a TIFF image into a numpy array."""
    return imread(path)


def show_side_by_side(img_full: np.ndarray, img_crop: np.ndarray, *, title_full: str, title_crop: str):
    """Display reference + crop with percentile normalization for contrast."""
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].imshow(img_full, cmap='gray', vmin=np.percentile(img_full, 1), vmax=np.percentile(img_full, 99))
    ax[0].set_title(title_full)
    ax[0].axis('off')

    ax[1].imshow(img_crop, cmap='gray', vmin=np.percentile(img_crop, 1), vmax=np.percentile(img_crop, 99))
    ax[1].set_title(title_crop)
    ax[1].axis('off')

    plt.tight_layout()
    plt.show()


def load_label_mask(path: Path) -> np.ndarray:
    """Load a precomputed segmentation mask (label image)."""
    return imread(path)


def show_segmentation_overlay(image: np.ndarray, mask: np.ndarray, title: str):
    """Overlay segmentation boundaries on an image for quick QA."""
    boundaries = find_boundaries(mask, mode='outer')
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(ij_percentile_normalize(image), cmap='gray')
    ax.imshow(np.ma.masked_where(~boundaries, boundaries), cmap='autumn', alpha=0.6)
    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# --- 1. Configuration ---

# File Paths (adjust to your data)
base_path = Path("path/to/your/data")
full_image_path = base_path / "reference_image.tif"
crop_image_path = base_path / "crop_image.tif"
output_dir = Path("nucleisky_output")
output_dir.mkdir(exist_ok=True)

# Pixel size metadata (µm/px). Prefer to read from TIFF metadata when possible.
pixel_size_full_um = get_pixel_size_um_from_tiff(full_image_path)
pixel_size_crop_um = get_pixel_size_um_from_tiff(crop_image_path)

# If metadata is missing, fall back to manual values.
if pixel_size_full_um is None:
    pixel_size_full_um = 0.65
if pixel_size_crop_um is None:
    pixel_size_crop_um = 0.65

# WARNING: 0.65 is a placeholder fallback for this demo only.
# Replace with acquisition-validated µm/px metadata before scientific use.

print(f"Output Directory: {output_dir.absolute()}")
print(f"Reference pixel size (µm/px): {pixel_size_full_um}")
print(f"Crop pixel size (µm/px): {pixel_size_crop_um}")


In [ ]:
# --- 2. Load & Display ---

print("Loading images...")
img_full = load_tiff(full_image_path)
img_crop = load_tiff(crop_image_path)

show_side_by_side(
    img_full,
    img_crop,
    title_full=f"Reference Image\n{img_full.shape}",
    title_crop=f"Crop (Moving)\n{img_crop.shape}",
)


In [ ]:
# --- 3. Segmentation ---

print("Normalizing scales for segmentation...")
(img_full_seg, img_crop_seg,
 ps_full_seg, ps_crop_seg,
 scale_f, scale_c, target_um) = scale_normalize_pair_for_segmentation(
    img_full, img_crop,
    pixel_size_full_um, pixel_size_crop_um,
)

# Define segmentation settings
seg_method = "threshold"  # Options: "threshold", "cellpose", "instanseg"
seg_settings = {
    "threshold": {
        "threshold_method": "otsu",
        "min_object_size": 5,
        "do_watershed": True,
    }
}

print(f"Segmenting using {seg_method}...")
masks_full = segment_nuclei_dispatch(img_full_seg, seg_method, ps_full_seg, seg_settings)
masks_crop = segment_nuclei_dispatch(img_crop_seg, seg_method, ps_crop_seg, seg_settings)

show_segmentation_overlay(img_full_seg, masks_full, "Reference segmentation")
show_segmentation_overlay(img_crop_seg, masks_crop, "Crop segmentation")

print("Extracting features...")
df_full = extract_nuclear_features(masks_full, None, ps_full_seg)
df_crop = extract_nuclear_features(masks_crop, None, ps_crop_seg)

# Map coordinates back to original pixel space
df_full = add_centroids_orig_px_columns(df_full, scale_f)
df_crop = add_centroids_orig_px_columns(df_crop, scale_c)

print(f"Found {len(df_full)} nuclei in Reference.")
print(f"Found {len(df_crop)} nuclei in Crop.")


## Matching Option A: Adaptive Pipeline

The `run_adaptive_matching_and_export` function is the *auto* mode. It automatically selects the best matching strategy (graph, quad, triangles, hashing) based on the number of nuclei, and retries with different strategies if the first one fails.


In [ ]:
# --- Option A: Adaptive Matching ---

print("Running Adaptive Pipeline...")
best_result_adaptive, history = run_adaptive_matching_and_export(
    df_full=df_full,
    df_crop=df_crop,
    img_full=img_full,
    img_crop=img_crop,
    pixel_size_full_um=pixel_size_full_um,
    pixel_size_crop_um=pixel_size_crop_um,
    result_dir=str(output_dir),
    store_full_out=True,
)

if best_result_adaptive['success']:
    print("\n✅ Adaptive Match Success!")
    print(f"Winning Matcher: {best_result_adaptive['matcher']}")
else:
    print("\n❌ Adaptive Match Failed.")


## Matching Option B: Manual Config (Advanced)

Use this when you want explicit control over matcher choice or configuration. For example:
- You know the rotation is small (e.g., ±10°), so checking ±180° is wasteful.
- You want to force a specific matcher (e.g., `quad`) because another matcher is too slow.

Below we manually configure the **Quad** matcher and restrict its rotation.


In [ ]:
# --- Option B: Manual Configuration ---

# 1. Customize configuration
custom_config = DEFAULT_MATCHER_CONFIG.copy()
custom_config['quad'] = custom_config.get('quad', {}).copy()

# Override: Restrict max rotation to +/- 20 degrees (default is 180)
custom_config['quad']['angle_max_deg'] = 20.0

# 2. Prepare data (extract µm coordinates)
centroids_f_um = extract_centroids_um(df_full, name="df_full")
centroids_c_um = extract_centroids_um(df_crop, name="df_crop")

print("Running Manual QUAD Matcher (Max Angle = 20°)...")

manual_result = NucleiSky(
    centroids_crop_um=centroids_c_um,
    centroids_full_um=centroids_f_um,
    img_full=img_full,
    img_crop=img_crop,
    ij_percentile_normalize=ij_percentile_normalize,
    pixel_size_full_um=pixel_size_full_um,
    pixel_size_crop_um=pixel_size_crop_um,
    matcher="quad",
    matcher_config=custom_config,
    df_full=df_full,
    df_crop=df_crop,
)

if manual_result['success']:
    print(f"\n✅ Manual Match Success!")
    print(f"Scale: {manual_result['best_scale']:.4f}")
    print(f"Translation (um): {manual_result['best_t']}")
    print(f"Inlier Fraction: {manual_result['match_quality']['frac_inliers']:.2f}")
else:
    print("\n❌ Manual Match Failed.")


In [ ]:
# --- 4. Visualize & Export ---

final_result = manual_result  # Or use best_result_adaptive

if final_result['success']:
    print("Generating visualization...")

    # 1. Plot overlay (display in notebook)
    show_alignment_original_and_rescaled(
        final_result,
        img_full_orig=img_full,
        img_crop_orig=img_crop,
        pixel_size_full_orig_um=pixel_size_full_um,
        pixel_size_crop_orig_um=pixel_size_crop_um,
        save_dir=output_dir,
    )
    plt.show()

    # 2. Save transform JSON (for reuse)
    json_path = output_dir / "manual_transform.json"
    save_nucleisky_transform(
        final_result,
        out_path=json_path,
        matcher_name=final_result['matcher'],
        pixel_size_full_um=pixel_size_full_um,
        pixel_size_crop_um=pixel_size_crop_um,
    )
    print(f"Saved transform: {json_path}")

    # 3. Export aligned ImageJ stacks (for inspection)
    print("Exporting ImageJ stacks...")
    export_aligned_imagej_stacks(
        final_result,
        out_dir=output_dir / "aligned_images",
        img_full=img_full,
        img_crop=img_crop,
        pixel_size_full_um=pixel_size_full_um,
        pixel_size_crop_um=pixel_size_crop_um,
        axes_full="YX",
        axes_crop="YX",
        export_region="full",  # or "roi" for just the crop area
        margin_px=50,
    )
    print(f"Export complete. Check folder: {output_dir / 'aligned_images'}")

else:
    print("Skipping export (no valid match found).")
